# Lab Day 2 — DeepWeeds: backbone, công thức huấn luyện, suy luận (Colab)

Notebook chạy **toàn bộ** bài lab bằng code đã hoàn thiện trong `code/`. Chạy lần lượt từ trên xuống.
Mọi kết quả (`runs/`, `predictions/`, `curves/`) được lưu vào **Google Drive**, nên nếu Colab bị ngắt chỉ cần
chạy lại các ô: lần chạy nào đã xong sẽ được bỏ qua.

**Quy tắc quan trọng:** chọn mọi thứ trên **val**; **test chỉ chạy một lần mỗi seed ở Bước 4**.
Runtime → Change runtime type → **GPU (T4)**.

## 0. Cài đặt

In [ ]:
!nvidia-smi -L
!pip -q install timm openpyxl
import sys, os, platform, json, time, glob, shutil
import numpy as np, pandas as pd, torch, timm
import matplotlib.pyplot as plt
print("python", platform.python_version(), "| torch", torch.__version__, "| timm", timm.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHÔNG CÓ GPU -> bật GPU rồi chạy lại")

### 0.1 Lấy code
Cách A (khuyên dùng): bạn đã `git push` repo lên GitHub → điền `REPO_URL`. Repo phải chứa thư mục `code/` và `eval.py`.
Cách B: không dùng git → nén cả thư mục repo thành `lab.zip`, tải lên Drive tại `MyDrive/lab_day2/lab.zip`,
đặt `USE_GIT = False`.

In [ ]:
USE_GIT = True
REPO_URL = "https://github.com/hieulovecat/K4-Track4-Day2-Deeplearning-Advance"   # sửa nếu cần
REPO_DIR = "/content/lab_repo"

from google.colab import drive
drive.mount("/content/drive")
WORK = "/content/drive/MyDrive/lab_day2"            # nơi lưu bền vững: runs, predictions, curves, results.xlsx
os.makedirs(WORK, exist_ok=True)

if USE_GIT:
    if os.path.exists(REPO_DIR):
        !git -C {REPO_DIR} pull -q
    else:
        !git clone -q {REPO_URL} {REPO_DIR}
else:
    !rm -rf {REPO_DIR} && mkdir -p {REPO_DIR} && unzip -q -o {WORK}/lab.zip -d {REPO_DIR}
    # nếu zip có thư mục con, trỏ REPO_DIR vào đúng thư mục chứa eval.py
    cand = [os.path.dirname(p) for p in glob.glob(f"{REPO_DIR}/**/eval.py", recursive=True)]
    REPO_DIR = cand[0]

CODE_DIR = f"{REPO_DIR}/code"
assert os.path.exists(f"{REPO_DIR}/eval.py") and os.path.exists(f"{CODE_DIR}/train.py"), "thiếu eval.py hoặc code/"
sys.path.insert(0, CODE_DIR)          # import train, model, dataset, ...
sys.path.insert(0, REPO_DIR)          # import eval
os.environ["EVAL_DIR"] = REPO_DIR
print("code:", CODE_DIR)

### 0.2 Dữ liệu
Ảnh: tải từ Zenodo vào đĩa cục bộ của Colab (nhanh hơn đọc từ Drive), kiểm tra MD5.
Nếu bạn đã có `images.zip` trên Drive, đặt `IMAGES_SOURCE = "drive"` và `DRIVE_ZIP`.
Nhãn và **fold 0** lấy nguyên bản từ GitHub của tác giả (không sửa).

In [ ]:
import hashlib, subprocess
IMAGES_SOURCE = "zenodo"                               # "zenodo" hoặc "drive"
DRIVE_ZIP = "/content/drive/MyDrive/images.zip"        # dùng khi IMAGES_SOURCE == "drive"
MD5 = "b7b30f96d466fba86016aa5a26606e0f"

os.makedirs("/content/data/labels", exist_ok=True)
zip_path = "/content/data/images.zip"
if not glob.glob("/content/data/**/*.jpg", recursive=True):
    if IMAGES_SOURCE == "zenodo":
        !wget -q -O {zip_path} "https://zenodo.org/records/7939060/files/images.zip?download=1"
    else:
        shutil.copy(DRIVE_ZIP, zip_path)
    h = hashlib.md5()
    with open(zip_path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    assert h.hexdigest() == MD5, f"MD5 sai: {h.hexdigest()}"
    !unzip -q -n {zip_path} -d /content/data/

BASE = "https://raw.githubusercontent.com/AlexOlsen/DeepWeeds/master/labels"
for name in ["labels", "train_subset0", "val_subset0", "test_subset0"]:
    if not os.path.exists(f"/content/data/labels/{name}.csv"):
        !wget -q -O /content/data/labels/{name}.csv {BASE}/{name}.csv

jpgs = glob.glob("/content/data/**/*.jpg", recursive=True)
IMAGES_DIR = os.path.dirname(jpgs[0]); LABELS_DIR = "/content/data/labels"
print(len(jpgs), "ảnh trong", IMAGES_DIR)

In [ ]:
EPOCHS = 12                         # GUIDE: 10-15; giữ giống nhau cho mọi thí nghiệm
NUM_WORKERS = min(4, os.cpu_count() or 2)
COMMON = dict(images_dir=IMAGES_DIR, labels_dir=LABELS_DIR, out_dir=f"{WORK}/runs",
              pred_dir=f"{WORK}/predictions", curves_dir=f"{WORK}/curves",
              num_workers=NUM_WORKERS, epochs=EPOCHS, batch_size=64)

import dataset as D, model as M, losses as L, train as TR, inference as INF, benchmark as BM
import checks, experiments as EX
from train import Config, run
import eval as ev
print({k: v for k, v in COMMON.items() if k.endswith("_dir") or k in ("epochs", "num_workers")})

## Bước 0 — EDA và kiểm tra pipeline

In [ ]:
# Kiểm tra chia dữ liệu (README 2.1): số ảnh, giao rỗng, hợp 17.509, file tồn tại. Dán kết quả vào báo cáo.
train_df, val_df, test_df = D.load_split(LABELS_DIR, fold=0)
info = D.check_split(train_df, val_df, test_df, IMAGES_DIR)
json.dump(info, open(f"{WORK}/split_check.json", "w"), indent=1)

In [ ]:
# Biểu đồ phân bố lớp + đối chiếu Table 1 của bài báo
tbl = pd.DataFrame(info["per_class"], index=D.CLASS_NAMES)
tbl["total"] = tbl.sum(axis=1)
paper = [1125, 1064, 1031, 1022, 1062, 1009, 1074, 1016, 9106]
tbl["paper_table1"] = paper
display(tbl)
tbl[["train", "val", "test"]].plot.bar(stacked=True, figsize=(9, 4), title="Số ảnh mỗi lớp (fold 0)")
plt.ylabel("số ảnh"); plt.tight_layout(); plt.savefig(f"{WORK}/eda_class_distribution.png", dpi=130); plt.show()
print("tỉ lệ lớp lớn nhất / nhỏ nhất:", tbl.total.max() / tbl.total.min())

In [ ]:
# >= 3 ảnh mỗi lớp
fig, axes = plt.subplots(9, 3, figsize=(7, 21))
for c in range(9):
    for j, (_, r) in enumerate(train_df[train_df.Label == c].sample(3, random_state=0).iterrows()):
        axes[c, j].imshow(plt.imread(f"{IMAGES_DIR}/{r.Filename}")); axes[c, j].axis("off")
    axes[c, 0].set_title(D.CLASS_NAMES[c], fontsize=9, loc="left")
plt.tight_layout(); plt.savefig(f"{WORK}/eda_samples.png", dpi=100); plt.show()

In [ ]:
# Kiểm tra pipeline (GUIDE 1.3): loss ban đầu ≈ ln 9, overfit 1 batch, ảnh sau augmentation
dev = TR.get_device()
tf_eval = D.build_transforms(False, 224)
probe = D.make_loader(train_df, IMAGES_DIR, D.build_transforms(True, 224, "basic"), 32, train=True, num_workers=2)
checks.initial_loss("resnet50", probe, dev)
checks.overfit_one_batch("resnet50", probe, dev)
checks.show_augmented(train_df, IMAGES_DIR, "basic", path=f"{WORK}/aug_check.png")

In [ ]:
# (Tuỳ chọn) chạy thử 1 epoch cho cả pipeline trước khi chạy thật: ~vài phút. Xoá kết quả smoke sau đó.
SMOKE = False
if SMOKE:
    t0 = time.time(); r = run(Config(exp_id="SMOKE", backbone="resnet50", **{**COMMON, "epochs": 1}))
    print("1 epoch resnet50:", round(time.time() - t0), "s ->", r["val_macro_f1"])
    shutil.rmtree(f"{WORK}/runs/SMOKE", ignore_errors=True)

## Bước 1 — So sánh 7 backbone (công thức nền T00, cùng seed 0)
Thời gian ước lượng: mỗi backbone vài phút đến ~15 phút trên T4 (CHƯA đo, hãy tự đo epoch đầu).

In [ ]:
for cfg in EX.backbone_configs(seed=0, **COMMON):
    run(cfg)

In [ ]:
runs = EX.collect_runs(COMMON["out_dir"])
bb = runs[runs.exp_id.str.startswith("B")].copy()
lat_bb = EX.backbone_latency([n for _, n in EX.BACKBONES])           # batch 1, FP32, 224
bb["lat_b1_p50_ms"] = bb.backbone.map(lambda n: lat_bb[n]["p50"])
cols = ["exp_id", "backbone", "weights_tag", "params_M", "gmacs", "val_macro_f1", "val_top1",
        "val_recall_chinee", "val_recall_snake", "train_time_per_epoch_s", "lat_b1_p50_ms", "best_epoch"]
display(bb[cols].sort_values("val_macro_f1", ascending=False).round(4))
plt.figure(figsize=(6, 4))
plt.scatter(bb.lat_b1_p50_ms, bb.val_macro_f1)
for _, r in bb.iterrows(): plt.annotate(r.exp_id, (r.lat_b1_p50_ms, r.val_macro_f1))
plt.xlabel("độ trễ batch 1 p50 (ms)"); plt.ylabel("macro-F1 val"); plt.grid(alpha=.3)
plt.title("Backbone: chất lượng vs độ trễ"); plt.tight_layout(); plt.savefig(f"{WORK}/backbone_tradeoff.png", dpi=130); plt.show()

## Bước 2 — Công thức huấn luyện
Chọn backbone đi tiếp dựa trên bảng ở Bước 1 (ghi lý do vào báo cáo), rồi chạy T00 và các ablation. Mỗi lần chỉ đổi **một** yếu tố.

In [ ]:
BEST_BACKBONE = "resnet50"      # <- ĐỔI theo kết quả Bước 1 (chọn trên val; có thể cân nhắc cả độ trễ)
T00 = EX.baseline_config(BEST_BACKBONE, seed=0, **COMMON)
run(T00)
for cfg, axis, desc in EX.training_ablations(BEST_BACKBONE, seed=0, **COMMON):
    print("\n>>", cfg.exp_id, axis, "|", desc)
    run(cfg)

In [ ]:
# Đo nhiễu: T00 với thêm 2 seed (cùng công thức) -> std. Chênh lệch nhỏ hơn std = "không phân biệt được".
for s in (1, 2):
    run(EX.baseline_config(BEST_BACKBONE, seed=s, **COMMON))
runs = EX.collect_runs(COMMON["out_dir"])
t00 = runs[runs.exp_id == "T00"].val_macro_f1
print(f"T00 macro-F1 val: mean {t00.mean():.4f}  std {t00.std(ddof=1):.4f}  (n={len(t00)})")
t = runs[runs.exp_id.str.startswith("T") & (runs.seed == 0)].copy()
base0 = t.loc[t.exp_id == "T00", "val_macro_f1"].iloc[0]
t["delta_vs_T00"] = t.val_macro_f1 - base0
t["> nhiễu (2·std)?"] = t.delta_vs_T00.abs() > 2 * t00.std(ddof=1)
display(t[["exp_id", "note", "val_macro_f1", "val_top1", "delta_vs_T00", "> nhiễu (2·std)?", "val_f1_chinee", "val_f1_snake",
           "train_time_per_epoch_s"]].round(4))

In [ ]:
# Kết hợp các yếu tố tốt (ví dụ): sửa theo bảng ở trên, rồi xem cộng dồn hay triệt tiêu
COMBO = Config(exp_id="T14", backbone=BEST_BACKBONE, seed=0, note="kết hợp: <mô tả>",
               # ví dụ: mix="cutmix", loss="ls", label_smoothing=0.1, ema_decay=0.999,
               **COMMON)
run(COMBO)

## Bước 3 — Suy luận (≥ 4 phương pháp) và độ trễ, trên **val**
`inference_study` chạy I00 (1 view), I01 (TTA lật), I02 (5-crop, 10-view), I03 (gộp logit), I04 (độ phân giải), I05 (ensemble),
I07 (temperature scaling), I08 (gộp BN, AMP, FP16) và đo độ trễ p50/p95/p99 ở batch 1.

In [ ]:
INF_SRC = ("T00", 0)            # <- run dùng cho nghiên cứu suy luận (thường là cấu hình tốt nhất trên val)
cfg_inf = EX.load_cfg(*INF_SRC, out_dir=COMMON["out_dir"], images_dir=IMAGES_DIR, labels_dir=LABELS_DIR,
                      pred_dir=COMMON["pred_dir"], curves_dir=COMMON["curves_dir"], num_workers=NUM_WORKERS)
others = [EX.load_cfg("T00", s, out_dir=COMMON["out_dir"], images_dir=IMAGES_DIR, labels_dir=LABELS_DIR,
                      pred_dir=COMMON["pred_dir"], num_workers=NUM_WORKERS) for s in (1, 2)]   # ensemble theo seed
inf_df = EX.inference_study(cfg_inf, ensemble_with=others)
display(inf_df.round(4))
inf_df.to_csv(f"{WORK}/inference_val.csv", index=False)

In [ ]:
plt.figure(figsize=(7, 4.5))
plt.scatter(inf_df.p50_ms, inf_df.val_macro_f1)
for _, r in inf_df.iterrows(): plt.annotate(r.exp_id, (r.p50_ms, r.val_macro_f1), fontsize=8)
plt.xlabel("độ trễ batch 1 p50 (ms)"); plt.ylabel("macro-F1 val"); plt.grid(alpha=.3)
plt.title("Suy luận: độ chính xác vs độ trễ"); plt.tight_layout(); plt.savefig(f"{WORK}/inference_tradeoff.png", dpi=130); plt.show()

# Bảng Latency: batch 1 và batch 32 cho các dtype (ghi rõ GPU/dtype/batch)
model_inf = TR.load_best_model(cfg_inf)
fused = INF.fuse_conv_bn(model_inf)
lat_rows = []
for dt in ("fp32", "amp", "fp16"):
    for b in (1, 32):
        lat_rows.append({"config": f"{INF_SRC[0]} {cfg_inf.backbone}", **BM.latency_report(model_inf, b, 224, dt, "cuda")})
for b in (1, 32):
    lat_rows.append({"config": f"{INF_SRC[0]} {cfg_inf.backbone} + gộp BN", **BM.latency_report(fused, b, 224, "fp32", "cuda", fused_bn=True)})
lat_df = pd.DataFrame(lat_rows); display(lat_df.round(2)); lat_df.to_csv(f"{WORK}/latency.csv", index=False)

## Bước 4 — Chung kết: ≥ 3 seed, **test đúng một lần mỗi seed**
1. **Chốt cấu hình trên val** (Bước 1–3): điền `FINAL_KW` và phương pháp suy luận `VIEWS/AGG/TS`.
2. Chạy `F01` (≥ 3 seed) và mốc `T00` + `I00` (cùng seed). Sau khi mở kết quả test **không** quay lại sửa cấu hình rồi chạy lại.

In [ ]:
# ĐIỀN SAU KHI CHỐT TRÊN VAL (ví dụ bên dưới chỉ là gợi ý, không phải kết luận):
FINAL_KW = dict(backbone=BEST_BACKBONE,
                # mix="cutmix", loss="ls", label_smoothing=0.1, ema_decay=0.999,
                note="cấu hình chung kết")
VIEWS, AGG, TS = "hflip", "logit", True      # suy luận chung kết: TTA lật, gộp logit, temperature scaling (T khớp trên val)
SEEDS = (0, 1, 2)

In [ ]:
# CHẠY TEST: một lần mỗi seed (file test đã tồn tại thì KHÔNG chạy lại)
EX.run_final("F01", seeds=SEEDS, views=VIEWS, agg=AGG, temperature=TS, **{**COMMON, **FINAL_KW})
EX.run_baseline(seeds=SEEDS, backbone=BEST_BACKBONE, **COMMON)       # mốc T00 + I00

In [ ]:
P = COMMON["pred_dir"]; TEST_CSV = f"{LABELS_DIR}/test_subset0.csv"; LABELS_CSV = f"{LABELS_DIR}/labels.csv"
for tag, pat in [("F01", f"{P}/F01_seed*_test.csv"), ("F01uncal", f"{P}/F01_uncal_seed*_test.csv"), ("T00", f"{P}/T00_seed*_test.csv")]:
    !python {REPO_DIR}/eval.py score --pred "{pat}" --test-csv {TEST_CSV} --labels {LABELS_CSV} --tag {tag} --out {WORK}/eval_out

In [ ]:
# Độ trễ cho I5: cấu hình dùng được cho thời gian thực (p95 <= 100 ms ở batch 1). Dùng đúng cấu hình đo độ trễ ở đây.
cfg_f = EX.load_cfg("F01", 0, out_dir=COMMON["out_dir"], images_dir=IMAGES_DIR, labels_dir=LABELS_DIR, pred_dir=P)
model_f = TR.load_best_model(cfg_f)
rt = BM.tta_latency(model_f, 2 if VIEWS == "hflip" else 1, batch_size=1, img_size=224, dtype="fp32", device="cuda")
print(rt)
LAT_P95 = rt["p95"]

In [ ]:
!python {REPO_DIR}/eval.py grade --final "{P}/F01_seed*_test.csv" --baseline "{P}/T00_seed*_test.csv" \
    --uncal "{P}/F01_uncal_seed*_test.csv" --final-val "{P}/F01_seed*_val.csv" \
    --latency-p95-ms {LAT_P95:.2f} --latency-method proper \
    --test-csv {TEST_CSV} --val-csv {LABELS_DIR}/val_subset0.csv --labels {LABELS_CSV} --out {WORK}/eval_out

In [ ]:
# Ma trận nhầm lẫn (tổng các seed) và ảnh bị đoán sai Chinee Apple <-> Snake Weed
g = ev.load_group(f"{P}/F01_seed*_test.csv", TEST_CSV)
cm = sum(m["confusion"] for m in g.metrics)
fig, ax = plt.subplots(figsize=(7, 6)); im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(9)); ax.set_yticks(range(9)); ax.set_xticklabels(D.CLASS_NAMES, rotation=60, ha="right"); ax.set_yticklabels(D.CLASS_NAMES)
for i in range(9):
    for j in range(9): ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=7, color="white" if cm[i, j] > cm.max() / 2 else "black")
ax.set(xlabel="dự đoán", ylabel="nhãn thật", title="F01 test, tổng các seed"); plt.tight_layout()
plt.savefig(f"{WORK}/confusion_F01.png", dpi=130); plt.show()

p0 = g.preds[0]
wrong = [(f, t, p) for f, t, p in zip(p0.filenames, p0.y_true, p0.y_pred) if {t, p} == {0, 7}][:8]
fig, axes = plt.subplots(2, 4, figsize=(11, 6))
for ax, (f, t, p) in zip(axes.ravel(), wrong):
    ax.imshow(plt.imread(f"{IMAGES_DIR}/{f}")); ax.axis("off")
    ax.set_title(f"thật: {D.CLASS_NAMES[t]}\nđoán: {D.CLASS_NAMES[p]}", fontsize=8)
plt.tight_layout(); plt.savefig(f"{WORK}/errors_chinee_snake.png", dpi=120); plt.show()

## Bước 5 — Sản phẩm: `results.xlsx`, gói nộp bài

In [ ]:
runs = EX.collect_runs(COMMON["out_dir"])
out = EX.build_results_xlsx(f"{WORK}/results.xlsx", runs, inf_df, lat_df,
                            {"F01": f"{FINAL_KW['backbone']} + công thức chung kết + {VIEWS}/{AGG}/TS={TS}",
                             "T00": "mốc: công thức nền + 1 view"},
                            LABELS_CSV, TEST_CSV, backbone_lat=lat_bb, pred_dir=P)
print("đã ghi", out)
pd.read_excel(out, sheet_name="Summary")

In [ ]:
# Gói bài nộp theo README mục 5 (tải về từ Drive, đặt trong submissions/<mssv>_<ho_ten>/)
MSSV_NAME = "mssv_ho_ten_khong_dau"                      # <- sửa
sub = f"{WORK}/submissions/{MSSV_NAME}"
os.makedirs(sub, exist_ok=True)
shutil.copy(f"{WORK}/results.xlsx", sub)
for d in ("curves", "predictions"):
    shutil.copytree(f"{WORK}/{d}", f"{sub}/{d}", dirs_exist_ok=True)
shutil.copytree(CODE_DIR, f"{sub}/code", dirs_exist_ok=True, ignore=shutil.ignore_patterns("__pycache__", "tests"))
shutil.copy(f"{REPO_DIR}/eval.py", f"{sub}/code/eval.py")
print(os.listdir(sub))
# Còn thiếu (tự viết): report.md (dàn ý GUIDE mục 6.3) và README.md riêng (link notebook này, version thư viện, seed)